# Fraud Detection Feature Engineering with EMR on EKS + RAPIDS

This notebook demonstrates feature engineering for fraud detection using EMR on EKS with NVIDIA RAPIDS acceleration.

In [ ]:
import os
import boto3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np

## Environment Setup

In [ ]:
# Get environment variables
VIRTUAL_CLUSTER_ID = os.environ.get('VIRTUAL_CLUSTER_ID')
EMR_EXECUTION_ROLE_ARN = os.environ.get('EMR_EXECUTION_ROLE_ARN')
S3_BUCKET = os.environ.get('S3_BUCKET')
AWS_REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-west-2')

print(f"Virtual Cluster ID: {VIRTUAL_CLUSTER_ID}")
print(f"S3 Bucket: {S3_BUCKET}")
print(f"Region: {AWS_REGION}")

## Option 1: Local Spark Session (for development/testing)

In [ ]:
# Create local Spark session for development
spark = SparkSession.builder \
    .appName("Fraud Detection Feature Engineering - Local") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## Option 2: Submit Job to EMR on EKS (for production)

In [ ]:
def submit_emr_job(script_path, job_name="fraud-feature-engineering"):
    """
    Submit a Spark job to EMR on EKS
    """
    emr_client = boto3.client('emr-containers', region_name=AWS_REGION)
    
    job_config = {
        'name': job_name,
        'virtualClusterId': VIRTUAL_CLUSTER_ID,
        'executionRoleArn': EMR_EXECUTION_ROLE_ARN,
        'releaseLabel': 'emr-7.9.0-latest',
        'jobDriver': {
            'sparkSubmitJobDriver': {
                'entryPoint': script_path,
                'sparkSubmitParameters': '--conf spark.rapids.sql.enabled=true --conf spark.plugins=com.nvidia.spark.SQLPlugin'
            }
        },
        'configurationOverrides': {
            'applicationConfiguration': [
                {
                    'classification': 'spark-defaults',
                    'properties': {
                        'spark.executor.instances': '4',
                        'spark.executor.memory': '30G',
                        'spark.executor.resource.gpu.amount': '1',
                        'spark.rapids.sql.enabled': 'true',
                        'spark.kubernetes.executor.podNamePrefix': 'fraud-detection'
                    }
                }
            ]
        }
    }
    
    response = emr_client.start_job_run(**job_config)
    job_run_id = response['id']
    
    print(f"EMR Job submitted: {job_run_id}")
    print(f"Monitor job: aws emr-containers describe-job-run --virtual-cluster-id {VIRTUAL_CLUSTER_ID} --id {job_run_id}")
    
    return job_run_id

# Example: Submit feature engineering job
# job_id = submit_emr_job(f's3://{S3_BUCKET}/fraud-data/feature_engineering.py')

## Sample Data Processing (Local Development)

In [ ]:
# Create sample fraud detection data
sample_data = [
    ("2024-01-01 10:00:00", "CUST_001", "TERM_001", 100.0, 0),
    ("2024-01-01 10:05:00", "CUST_001", "TERM_002", 5000.0, 1),
    ("2024-01-01 11:00:00", "CUST_002", "TERM_001", 50.0, 0),
    ("2024-01-01 11:30:00", "CUST_002", "TERM_003", 200.0, 0),
]

schema = StructType([
    StructField("TX_DATETIME", StringType(), True),
    StructField("CUSTOMER_ID", StringType(), True),
    StructField("TERMINAL_ID", StringType(), True),
    StructField("TX_AMOUNT", DoubleType(), True),
    StructField("TX_FRAUD", IntegerType(), True)
])

df = spark.createDataFrame(sample_data, schema)
df.show()

In [ ]:
# Feature engineering example
from pyspark.sql.window import Window

# Convert timestamp
df = df.withColumn("TX_DATETIME", F.to_timestamp("TX_DATETIME"))

# Extract time features
df = df.withColumn("hour", F.hour("TX_DATETIME")) \
       .withColumn("day_of_week", F.dayofweek("TX_DATETIME"))

# Customer transaction frequency in last 15 minutes
window_spec = Window.partitionBy("CUSTOMER_ID") \
                   .orderBy("TX_DATETIME") \
                   .rangeBetween(-900, 0)  # 15 minutes in seconds

df = df.withColumn("customer_tx_count_15min", F.count("*").over(window_spec)) \
       .withColumn("customer_avg_amount_15min", F.avg("TX_AMOUNT").over(window_spec))

df.show()

## Save Results to S3

In [ ]:
# Save processed features to S3
output_path = f"s3://{S3_BUCKET}/fraud-data/processed-features/"

df.write \
  .mode("overwrite") \
  .option("path", output_path) \
  .saveAsTable("fraud_features")

print(f"Features saved to: {output_path}")

## Next Steps

1. **Scale Up**: Use the EMR job submission function to process large datasets
2. **GPU Acceleration**: Enable RAPIDS for faster processing on GPU nodes
3. **Pipeline Integration**: Use this notebook as part of an Argo Workflow
4. **Model Training**: Move to Ray ML notebook for distributed training

In [ ]:
# Clean up
spark.stop()